# 04 — Xuất model YOLO (.pt) sang ONNX (.onnx)

**Mục tiêu:** chuyển `best.pt` (bất kỳ notebook train nào — độ chín hay bệnh lá) sang `.onnx` để chạy trong trình duyệt với `ai/scripts/leaf_disease_tester.html` (hoặc công cụ tương tự cho nhánh độ chín sau này). Trình duyệt không chạy được `.pt` (PyTorch) trực tiếp, phải qua ONNX.

Notebook này **dùng chung cho cả 2 nhánh** — không hardcode class cụ thể, chỉ xuất định dạng.

## Trước khi chạy
1. **Add Input** → Kaggle Dataset chứa file `.pt` cần xuất (vd output đã Save Version của `03b_train_leaf_augmented.ipynb` hoặc `02c_train_ripeness_augmented.ipynb`).
2. Không cần GPU. Internet: **ON** (cần cài `onnxslim` để tối ưu file ONNX).
3. Nếu Kaggle Dataset chứa nhiều file `.pt` (vd cả `best.pt` lẫn `last.pt`), notebook sẽ liệt kê hết — xem kỹ Bước 1 trước khi để nó tự chọn, hoặc tự đặt `PT_OVERRIDE`.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics", "onnx", "onnxslim"], check=False)

### Bước 1 — Liệt kê toàn bộ file `.pt` tìm thấy trong `/kaggle/input`
Không đoán file nào là đúng — in hết ra kèm kích thước + đường dẫn đầy đủ để tự xác nhận trước khi export.

In [ ]:
from pathlib import Path

all_pt_files = sorted(Path("/kaggle/input").rglob("*.pt"))

if not all_pt_files:
    print("Các thư mục cấp 1 trong /kaggle/input:")
    for p in sorted(Path("/kaggle/input").glob("*")):
        print(" -", p)
    raise FileNotFoundError(
        "Không tìm thấy file .pt nào trong /kaggle/input. Kiểm tra đã Add Input đúng Kaggle Dataset "
        "chứa model đã train (đã Save Version) chưa."
    )

print(f"Tìm thấy {len(all_pt_files)} file .pt:\n")
for p in all_pt_files:
    size_mb = p.stat().st_size / 1e6
    print(f"  [{size_mb:6.1f} MB] {p}")

# Đặt thủ công nếu muốn chọn khác với lựa chọn tự động bên dưới, vd:
# PT_OVERRIDE = Path("/kaggle/input/notebooks/.../models/tomato_leaf_disease_yolov8n_augmented.pt")
PT_OVERRIDE = None

### Bước 2 — Chọn file để export
Ưu tiên: `PT_OVERRIDE` (nếu đặt) → file nằm trong thư mục `models/` (bản sao đặt tên rõ ràng, do các notebook train tự lưu) → file tên chứa "best" (không lấy "last.pt", vốn là checkpoint cuối cùng chứ không phải tốt nhất) → nếu vẫn không rõ, lấy file đầu tiên và cảnh báo kiểm tra lại.

In [ ]:
def pick_pt_file(files):
    in_models_dir = [p for p in files if p.parent.name == "models"]
    if in_models_dir:
        return in_models_dir[0], "nam trong thu muc models/"
    best_named = [p for p in files if "best" in p.stem.lower()]
    if best_named:
        return best_named[0], "ten file chua 'best'"
    return files[0], "khong ro tieu chi, lay file dau tien -> KIEM TRA LAI cho chac"


if PT_OVERRIDE is not None:
    PT_PATH = PT_OVERRIDE
    reason = "PT_OVERRIDE do nguoi dung dat"
else:
    PT_PATH, reason = pick_pt_file(all_pt_files)

print(f"Đã chọn: {PT_PATH}")
print(f"Lý do: {reason}")
print(f"Kích thước: {PT_PATH.stat().st_size / 1e6:.1f} MB")

### Bước 3 — Export sang ONNX

In [ ]:
import shutil

from ultralytics import YOLO

# /kaggle/input chỉ đọc — model.export() lại mặc định lưu file .onnx ngay cạnh file .pt nguồn,
# nên phải copy .pt sang /kaggle/working (ghi được) trước rồi mới load + export từ đó.
WORKING_PT = Path("/kaggle/working") / PT_PATH.name
shutil.copy2(PT_PATH, WORKING_PT)

model = YOLO(str(WORKING_PT))
print("Class trong model:", model.names)

onnx_path = model.export(format="onnx", imgsz=640, simplify=True)
onnx_path = Path(onnx_path)
print(f"\nĐã xuất: {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB)")